# Isolation Forest Training

Trains one Isolation Forest per OCR model backbone. Each model's frozen CNN is used as a feature extractor over the CAPTCHA training set, producing a (N, 256) feature matrix via global average pooling. The Isolation Forest is then fit on these features and saved to `weights/isolation_forest/`.

In [1]:
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import torch
from dotenv import find_dotenv, load_dotenv
from sklearn.ensemble import IsolationForest
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

load_dotenv(find_dotenv())

PROJECT_ROOT = Path(os.environ["PROJECT_ROOT_DIR"])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.kaggledataset import KaggleDataset
from src.models.isolation_forest.feature_extractor import CaptchaFeatureExtractor

## Configuration

In [2]:
SEED        = 42
NUM_SAMPLES = 50_000   # Subsample from full HF dataset — enough for Isolation Forest
BATCH_SIZE  = 256

MODEL_NAMES = [
    "crnn_base",
    "crnn_finetuned",
    "convtrans_base",
    "convtrans_finetuned",
]

WEIGHTS_DIR = PROJECT_ROOT / "weights" / "isolation_forest"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

## Dataset

Loads from `data/hammer_captchas/` (HuggingFace CAPTCHA images) for real training, or falls back to `data/cifar10/` for pipeline testing if the HuggingFace dataset hasn't been downloaded yet.

In [3]:
def collate_images_only(batch):
    """Collate images only, discarding variable-length label tensors."""
    images = torch.stack([item[0] for item in batch])
    return (images,)

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((40, 150)),
    transforms.ToTensor(),
])

captcha_dir = PROJECT_ROOT / "data" / "hammer_captchas"
fallback_dir = PROJECT_ROOT / "data" / "cifar10"

if any(captcha_dir.glob("*.jpg")):
    from src.datasets.kaggledataset import KaggleDataset
    full_dataset = KaggleDataset(root_dir=str(captcha_dir), transform=transform, preload=False)
    print(f"Loaded HuggingFace CAPTCHA dataset: {len(full_dataset):,} images")
else:
    from src.datasets.cifar10dataset import CIFAR10Dataset
    full_dataset = CIFAR10Dataset(root_dir=str(fallback_dir), transform=transform)
    print(f"WARNING: hammer_captchas/ empty — using CIFAR-10 as fallback for pipeline testing")
    print(f"Loaded CIFAR-10 dataset: {len(full_dataset):,} images")

random.seed(SEED)
indices = random.sample(range(len(full_dataset)), min(NUM_SAMPLES, len(full_dataset)))
dataset = Subset(full_dataset, indices)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_images_only,
)

print(f"Training Isolation Forest on {len(dataset):,} images")

Loaded HuggingFace CAPTCHA dataset: 1,365,874 images
Training Isolation Forest on 50,000 images


## Feature Extraction + Isolation Forest Fitting

For each model:
1. Load the frozen CNN backbone from HuggingFace
2. Extract (N, 256) feature matrix via global average pooling
3. Fit Isolation Forest
4. Save to `weights/isolation_forest/{model_name}.joblib`

In [4]:
for model_name in MODEL_NAMES:
    print(f"\n{'=' * 60}")
    print(f"  {model_name}")
    print(f"{'=' * 60}")

    # --- Feature extraction ---
    extractor = CaptchaFeatureExtractor(model_name)
    features = extractor.extract(dataloader)
    print(f"Feature matrix: {features.shape}  (mean={features.mean():.4f}, std={features.std():.4f})")

    # --- Isolation Forest ---
    print("Fitting Isolation Forest...")
    clf = IsolationForest(
        n_estimators=100,
        contamination="auto",
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(features)

    # --- Save ---
    save_path = WEIGHTS_DIR / f"{model_name}.joblib"
    joblib.dump(clf, save_path)
    print(f"Saved → {save_path}")

print("\nAll models trained and saved.")


  crnn_base
Loading Graf-J/captcha-crnn-base on cuda...


W0309 20:03:32.636000 22416 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/38 [00:00<?, ?it/s]

Extracting [crnn_base]: 100%|██████████| 196/196 [00:48<00:00,  4.06it/s]


Feature matrix: (50000, 256)  (mean=-0.0687, std=0.0902)
Fitting Isolation Forest...
Saved → C:\Users\jackb\Documents\MLDS\deep-learning\final_proj\Captcha-Recognition\weights\isolation_forest\crnn_base.joblib

  crnn_finetuned
Loading Graf-J/captcha-crnn-finetuned on cuda...


...

Feature matrix: (50000, 256)  (mean=0.0067, std=0.4810)
Fitting Isolation Forest...
Saved → C:\Users\jackb\Documents\MLDS\deep-learning\final_proj\Captcha-Recognition\weights\isolation_forest\convtrans_base.joblib

  convtrans_finetuned
Loading Graf-J/captcha-conv-transformer-finetuned on cuda...


C:\Users\jackb\.cache\huggingface\modules\transformers_modules\Graf_hyphen_J\captcha_hyphen_conv_hyphen_transformer_hyphen_finetuned\4bc31fa679c5f627e56dfa820ea7edad8cd5981a\modeling_captcha.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Loading weights:   0%|          | 0/43 [00:00<?, ?it/s]

Extracting [convtrans_finetuned]: 100%|██████████| 196/196 [00:28<00:00,  6.98it/s]


Feature matrix: (50000, 256)  (mean=0.0346, std=0.4958)
Fitting Isolation Forest...
Saved → C:\Users\jackb\Documents\MLDS\deep-learning\final_proj\Captcha-Recognition\weights\isolation_forest\convtrans_finetuned.joblib

All models trained and saved.
